In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.preprocessing import LabelEncoder

In [3]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
from xgboost import XGBClassifier
# from sklearn.model_selection import cross_val_score

In [5]:
# from sklearn.tree import DecisionTreeClassifier

In [6]:
df=pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
# df.info()

In [7]:
Pre_X=df.iloc[:, :-1]
Pre_y=df.iloc[:,-1]

# Select numerical features (int64 and float64)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Select categorical features (object and category)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
# print(Pre_y.head())
# print(Pre_X.head())

In [8]:
le = LabelEncoder()
y_encoded = le.fit_transform(Pre_y)

In [9]:
trf1 = ColumnTransformer([
    ('ohe1', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1,11,12]),
    ('ohe2', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [13,14,15]),
    ('ohe3', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [17,19])
],
remainder='passthrough')

In [10]:
# trf1.fit_transform(Pre_X).shape

In [11]:
trf2 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0, 44))
], remainder='passthrough')

In [12]:
trf3 = ColumnTransformer([
    ('select', SelectKBest(chi2, k=40), slice(0, 44))
], remainder='passthrough')

In [13]:
pipe=Pipeline([('trf1',trf1),('trf2',trf2),('trf3',trf3)])
df_processed=pipe.fit_transform(Pre_X,y_encoded)

In [14]:
df_tf=pd.DataFrame(df_processed,columns=pipe.get_feature_names_out())

In [15]:
def objective(trial):
    # Split data to validate the hyperparameters
    train_x, val_x, train_y, val_y = train_test_split(df_tf, y_encoded, test_size=0.2, random_state=42)

    # Define the search space
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
    }

    # Initialize and fit model
    model = XGBClassifier(**param, random_state=42, use_label_encoder=True, eval_metric='mlogloss')
    model.fit(train_x, train_y)

    # Predict and calculate accuracy
    preds = model.predict(val_x)
    accuracy = accuracy_score(val_y, preds)
    
    return accuracy

In [16]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5) # You can increase n_trials for better results

print("Best trial:")
trial = study.best_trial
print(f"  Value: {trial.value}")
print("Params: ")
for key, value in trial.params.items():
    print(f"{key}: {value}")

[I 2026-04-12 10:10:15,765] A new study created in memory with name: no-name-a4ba1b00-9939-4e28-ac23-d9fc437df9db
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:10:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-04-12 10:10:39,485] Trial 0 finished with value: 0.981579365079365 and parameters: {'n_estimators': 315, 'max_depth': 4, 'learning_rate': 0.017495212868069886, 'subsample': 0.962783917938584, 'colsample_bytree': 0.6611086148203115, 'gamma': 1.2784842442435913e-08}. Best is trial 0 with value: 0.981579365079365.
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:10:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-04-12 10:11:15,505] Trial 1 finished with value: 0.9842777777777778 and parameters: {'n

Best trial:
  Value: 0.9845396825396825
Params: 
n_estimators: 203
max_depth: 6
learning_rate: 0.08450073694332534
subsample: 0.7214250381976215
colsample_bytree: 0.9822194041051167
gamma: 0.0020406800573443988


In [17]:
# df_tf.info()

In [18]:
df2=pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
p_test=pipe.transform(df2)

In [19]:
x_text=pd.DataFrame(p_test,columns=pipe.get_feature_names_out())

In [20]:
x_text.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270000 entries, 0 to 269999
Data columns (total 40 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   select__scale__ohe1__Soil_Type_Clay                270000 non-null  float64
 1   select__scale__ohe1__Soil_Type_Loamy               270000 non-null  float64
 2   select__scale__ohe1__Soil_Type_Sandy               270000 non-null  float64
 3   select__scale__ohe1__Soil_Type_Silt                270000 non-null  float64
 4   select__scale__ohe1__Crop_Type_Cotton              270000 non-null  float64
 5   select__scale__ohe1__Crop_Type_Maize               270000 non-null  float64
 6   select__scale__ohe1__Crop_Type_Potato              270000 non-null  float64
 7   select__scale__ohe1__Crop_Type_Rice                270000 non-null  float64
 8   select__scale__ohe1__Crop_Type_Sugarcane           270000 non-null  float6

In [21]:
# model=DecisionTreeClassifier()
# model.fit(df_tf,y_encoded)

In [22]:
# pred=model.predict(x_text)
# print(pred)

In [23]:
best_model = XGBClassifier(**study.best_params)
best_model.fit(df_tf, y_encoded)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9822194041051167, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=0.0020406800573443988, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.08450073694332534, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=203, n_jobs=None,
              num_parallel_tree=None, ...)

In [24]:
pred2 = best_model.predict(x_text)

In [25]:
sub=pd.DataFrame(df2['id'])
sub['Irrigation_Need']=le.inverse_transform(pred2)
print(sub)

            id Irrigation_Need
0       630000             Low
1       630001             Low
2       630002             Low
3       630003             Low
4       630004             Low
...        ...             ...
269995  899995          Medium
269996  899996             Low
269997  899997          Medium
269998  899998             Low
269999  899999          Medium

[270000 rows x 2 columns]


In [26]:
sub.to_csv('submission.csv', index=False)